# PRISM embedding inventory
Run on Colab with Drive mounted. Writes `PRISM/embedding_inventory.csv` (path, shape, dtype, size, SHA256) and prints a per-(model, dataset) summary. Read-only: nothing is modified or uploaded.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, hashlib, numpy as np, pandas as pd
ROOT = '/content/drive/MyDrive/PRISM'
TREES = {'main': f'{ROOT}/embeddings', 'camelyon17': f'{ROOT}/embeddings_camelyon17'}

def sha256(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

rows = []
for tree, base in TREES.items():
    if not os.path.isdir(base):
        print('missing:', base); continue
    for dirpath, _, files in os.walk(base):
        for fn in sorted(files):
            if not fn.endswith('.npy'):
                continue
            p = os.path.join(dirpath, fn)
            rel = os.path.relpath(p, base).split(os.sep)
            arr = np.load(p, mmap_mode='r')
            rows.append(dict(tree=tree, model=rel[0], group=rel[1] if len(rel) > 2 else '',
                             file=fn, shape=str(arr.shape), dtype=str(arr.dtype),
                             size_mb=round(os.path.getsize(p) / 2**20, 2), sha256=sha256(p),
                             path=os.path.relpath(p, ROOT)))
            print(f"{tree:10s} {rel[0]:12s} {rows[-1]['group']:14s} {fn:22s} {rows[-1]['shape']:>18s} {rows[-1]['size_mb']:>9.1f} MB")

inv = pd.DataFrame(rows)
inv.to_csv(f'{ROOT}/embedding_inventory.csv', index=False)
print('\nTotal GB by tree:', (inv.groupby('tree').size_mb.sum() / 1024).round(2).to_dict())
print(inv.pivot_table(index='model', columns='group', values='size_mb', aggfunc='sum').round(0).to_string())
print('\nSaved:', f'{ROOT}/embedding_inventory.csv')